# 第 2 節｜BERT 二元分類微調：臺灣繁體中文 SST-2 影評

對應 HTML 投影片第 7～12 頁。資料**只使用提供的 `sst2_train_zh_TW.csv`**，不混入其他來源。

| 欄位 | 意義 | 是否給 BERT 當輸入 |
|---|---|---|
| `idx` | 影評編號 | 否 |
| `sentence` | 臺灣繁體中文影評 | 是 |
| `label` | 情緒答案，`0=負面`、`1=正面` | 訓練標籤 y（不是文字輸入） |

完整 CSV 有 **67,349 筆**。預設使用 **10,000 筆**不重複影評練習（Train 6,400／Validation 1,600／Test 2,000）。將 `SAMPLE_SIZE = 10000` 改成 `SAMPLE_SIZE = None`，就會使用**原始完整 67,349 筆**。完整模式保留重複影評，但資料切分會讓同一句文字只屬於其中一組，避免資料洩漏。完整微調會明顯增加 GPU 訓練時間。

**全部 Code Cell 都已填好程式，學生只需修改參數並執行。** 可調整 `SAMPLE_SIZE`、`MODEL_ID`、`EPOCHS`、`BATCH_SIZE`、`LEARNING_RATE`，也可以在最後改寫新影評。

**訓練流程：** 讀取 CSV → 抽樣 → Train／Validation／Test → Tokenizer → BERT 微調 → 兩條 Loss → Test → 新影評預測。

## 準備環境｜先將 CSV 上傳到 Colab 工作目錄

把 `sst2_train_zh_TW.csv` 放在目前 Notebook 的工作目錄（Colab 預設為 `/content`），選擇 T4 GPU，執行下一格。`pd.read_csv()` 讀取的是遠端執行環境內的檔案。

In [ ]:
# Install the packages required for BERT fine-tuning.
# 安裝 BERT 微調需要的套件。
%pip -q install transformers==4.57.1 datasets==3.6.0 accelerate sentencepiece scikit-learn

# Import tools for reading data and drawing learning curves.
# 匯入資料讀取與訓練曲線繪圖工具。
import pandas as pd

# Import the plotting library.
# 匯入繪製 Loss 曲線的函式庫。
import matplotlib.pyplot as plt

# Import the framework used for model training.
# 匯入模型訓練使用的 PyTorch。
import torch

# Import the data conversion utility.
# 匯入 Hugging Face 的資料集轉換工具。
from datasets import Dataset

# Import train/test splitting and evaluation metrics.
# 匯入資料切分與分類評估工具。
from sklearn.model_selection import GroupShuffleSplit

# Import the evaluation helpers.
# 匯入模型評估指標。
from sklearn.metrics import accuracy_score, classification_report

# Import the tokenizer, classifier, and training tools.
# 匯入 Tokenizer、分類模型與微調工具。
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, Trainer, TrainingArguments, pipeline

# Load only the provided Taiwanese Traditional Chinese SST-2 CSV.
# 只讀取提供的臺灣繁體中文 SST-2 CSV。
df = pd.read_csv("sst2_train_zh_TW.csv")

# Remove only rows without training text or a valid label.
# 只移除缺少文字或標籤的資料列。
df = df.dropna(subset=["sentence", "label"])

# Convert the class labels into integers.
# 將正負面標籤轉成整數。
df["label"] = df["label"].astype(int)

# Set the number of classroom reviews; None uses all 67,349 original rows.
# 設定課堂使用筆數；改成 None 即使用原始完整 67,349 筆。
SAMPLE_SIZE = 10000

# Sample unique sentences for the smaller teaching run.
# 小規模教學抽樣先移除重複句子，避免資料洩漏。
if SAMPLE_SIZE is not None:
    # Remove repeated texts before taking a reproducible subset.
    # 先保留不重複的句子，再固定抽取指定筆數。
    df = df.drop_duplicates(subset=["sentence"]).sample(n=min(SAMPLE_SIZE, df["sentence"].nunique()), random_state=42)

# Choose the pretrained Chinese BERT checkpoint.
# 指定要微調的預訓練中文 BERT 模型。
MODEL_ID = "google-bert/bert-base-chinese"

# Check the class distribution before splitting.
# 切分資料前先檢查正負面筆數。
print("本次使用資料筆數：", len(df), "｜負面 0 / 正面 1：", df["label"].value_counts().sort_index().to_dict())


### 範例 1｜切分資料並建立 BERT 的輸入

直接執行下一格程式，不需要複製或自己撰寫。預設 10,000 筆：**Train 6,400／Validation 1,600／Test 2,000**。

如需調整資料筆數，只修改第一格的 `SAMPLE_SIZE`：例如 `5000`、`10000`，或使用 `None` 代表完整 67,349 筆。完整模式依句子分組切分，三份筆數可能因重複句子的分組而略有差異。

`GroupShuffleSplit` 會避免同一句 `sentence` 同時出現在不同資料集。Tokenizer 只讀取 `sentence`，`label` 則用於訓練和評估。

In [ ]:
# Split by sentence so identical reviews cannot cross data partitions.
# 依照影評句子分組切分，避免相同文字同時出現在不同資料集。
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

# Reserve the held-out test set before selecting a model.
# 在選擇模型前先保留獨立測試集。
train_indices, test_indices = next(splitter.split(df, groups=df["sentence"]))

# Separate the initial training and test partitions.
# 分開初步切分的訓練與測試資料。
train_df, test_df = df.iloc[train_indices], df.iloc[test_indices]

# Split validation from training without repeating a sentence across groups.
# 再從訓練資料分出驗證集，避免相同句子跨組。
train_indices, val_indices = next(splitter.split(train_df, groups=train_df["sentence"]))

# Keep the final training and validation partitions separate.
# 分別保留最終的訓練與驗證資料。
train_df, val_df = train_df.iloc[train_indices], train_df.iloc[val_indices]

# Load the tokenizer matching the selected pretrained BERT checkpoint.
# 載入與所選預訓練 BERT 模型相符的 Tokenizer。
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Define one text-processing rule shared by all partitions.
# 定義所有資料集共用的文字處理規則。
def encode(batch):
    # Tokenize review sentences and truncate overly long inputs.
    # 將影評轉成 Token，並截斷過長的輸入。
    return tokenizer(batch["sentence"], truncation=True, max_length=96)

# Convert the training partition into a Hugging Face dataset.
# 將訓練資料轉成 Hugging Face Dataset。
train_data = Dataset.from_pandas(train_df, preserve_index=False).map(encode, batched=True)

# Convert the validation partition using the same tokenizer.
# 使用相同 Tokenizer 轉換驗證資料。
val_data = Dataset.from_pandas(val_df, preserve_index=False).map(encode, batched=True)

# Convert the test partition without using it for model selection.
# 轉換測試資料，但不拿來選擇模型。
test_data = Dataset.from_pandas(test_df, preserve_index=False).map(encode, batched=True)

# Display the exact numbers of examples in each partition.
# 顯示三份資料實際包含的筆數。
print("Train:", len(train_data), "Validation:", len(val_data), "Test:", len(test_data))

### 範例 2｜調整超參數並微調 BERT

直接執行下一格程式。**不需要自己撰寫訓練程式**，只要修改開頭這三個值，就能比較訓練結果：

| 參數 | 預設值 | 用途 |
|---|---:|---|
| `EPOCHS` | 3 | 訓練資料重複學習幾輪 |
| `BATCH_SIZE` | 16 | 每次訓練讀取幾筆影評 |
| `LEARNING_RATE` | `3e-5` | 每次更新模型權重的幅度 |

每輪都會記錄 Train／Validation Loss，並以最低 Validation Loss 選擇最佳 Checkpoint。第一次執行需要下載模型；使用完整資料會增加訓練時間。

In [ ]:
# Set the number of complete training passes.
# 設定要將訓練資料完整學習幾輪。
EPOCHS = 3

# Set how many reviews each training batch contains.
# 設定每個訓練 Batch 包含多少筆影評。
BATCH_SIZE = 16

# Set the learning rate used to update the model weights.
# 設定更新模型權重時使用的 Learning Rate。
LEARNING_RATE = 3e-5

# Load the pretrained BERT with a new two-class classification head.
# 載入預訓練 BERT，並建立二元分類頭。
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID, num_labels=2)

# Configure training, validation, and saving of the best checkpoint.
# 設定訓練、驗證與儲存最佳 Checkpoint 的方式。
settings = TrainingArguments(
    output_dir="03_bert_sst2_output", num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=32,
    learning_rate=LEARNING_RATE, logging_strategy="epoch", eval_strategy="epoch",
    save_strategy="epoch", save_total_limit=2,
    load_best_model_at_end=True, metric_for_best_model="eval_loss",
    fp16=torch.cuda.is_available(), report_to="none"
)

# Create a trainer using dynamic padding for each text batch.
# 建立微調工具，讓每個 Batch 依實際句長動態補齊。
trainer = Trainer(model=model, args=settings, train_dataset=train_data, eval_dataset=val_data, data_collator=DataCollatorWithPadding(tokenizer), processing_class=tokenizer)

# Fine-tune the BERT encoder and classification head.
# 微調 BERT Encoder 與分類頭。
trainer.train()

# Save the checkpoint selected by validation loss.
# 儲存依照 Validation Loss 選出的最佳 Checkpoint。
trainer.save_model(settings.output_dir)

# Save the tokenizer beside the trained model.
# 將 Tokenizer 一起儲存在模型目錄。
tokenizer.save_pretrained(settings.output_dir)

## Optional 1｜比較真實的 Train 與 Validation Loss

直接執行下方程式，從 `trainer.state.log_history` 讀取剛才訓練的實際紀錄，並畫出兩條 Loss 曲線。不需自行寫圖表程式。

In [ ]:
# Turn the trainer's real training log into a table.
# 將實際訓練紀錄整理為表格。
history = pd.DataFrame(trainer.state.log_history)

# Select the training loss logged at each epoch.
# 取出每個 Epoch 的 Train Loss。
train_history = history.dropna(subset=["loss"])

# Select the validation loss logged at each epoch.
# 取出每個 Epoch 的 Validation Loss。
val_history = history.dropna(subset=["eval_loss"])

# Draw the actual training-loss curve.
# 繪製實際的 Train Loss 曲線。
plt.plot(train_history["epoch"], train_history["loss"], marker="o", label="Train Loss")

# Draw the actual validation-loss curve for comparison.
# 繪製實際的 Validation Loss 曲線以便比較。
plt.plot(val_history["epoch"], val_history["eval_loss"], marker="o", label="Validation Loss")

# Identify the horizontal and vertical axes.
# 標示圖表的橫軸與縱軸。
plt.xlabel("Epoch")

# Label the vertical axis with the measured loss.
# 標示縱軸為實際測得的 Loss。
plt.ylabel("Loss")

# Show the learning curves and their labels.
# 顯示兩條學習曲線與圖例。
plt.legend()

# Display the complete chart.
# 顯示完整的圖表。
plt.show()

# Show the best model chosen using validation, not the test set.
# 顯示使用驗證集（不是測試集）選出的最佳模型。
print("最佳 Checkpoint：", trainer.state.best_model_checkpoint)


## Optional 2｜測試模型並預測一句新影評

直接執行下方程式即可顯示 Test Accuracy、Precision、Recall、F1。想測試自己的文字時，只需修改 `text` 那一句；其他句子已放在程式註解內供複製。

最後才使用 Test 評估最佳微調模型。標籤 `LABEL_0=負面`、`LABEL_1=正面`；模型分數不保證分類一定正確。

In [ ]:
# Predict the reviews that were reserved for the final test.
# 預測先前保留、沒有參與訓練的測試影評。
test_result = trainer.predict(test_data)

# Select the class with the largest output logit.
# 以最高 Logit 對應的類別作為預測結果。
predicted = test_result.predictions.argmax(axis=1)

# Measure accuracy on the untouched test partition.
# 計算最後測試資料的 Accuracy。
print("Test Accuracy：", round(accuracy_score(test_df["label"], predicted), 3))

# Show precision, recall, and F1 for both review labels.
# 顯示負面與正面兩類的 Precision、Recall 與 F1。
print(classification_report(test_df["label"], predicted, labels=[0, 1], target_names=["負面", "正面"], digits=3))

# Choose one new review for prediction.
# 選一句新的影評進行預測。
text = "這部電影節奏明快，演員表現很出色。"

# Copy either sentence to replace the one above when practicing.
# 練習時可複製其中一句，替換上面的文字。
# text = "劇情拖沓又無聊，我差點睡著。"
# text = "有幾個笑點，但結尾有點可惜。"

# Create the inference pipeline using the best fine-tuned model.
# 使用驗證最佳模型建立推論工具。
classifier = pipeline("text-classification", model=trainer.model, tokenizer=tokenizer, device=0 if torch.cuda.is_available() else -1)

# Classify the sentence with the best fine-tuned model.
# 使用最佳微調模型判斷這句影評。
result = classifier(text, truncation=True, max_length=96)[0]

# Print the original label code and model score without a manual lookup.
# 直接輸出原始分類代碼及分數，不另建手動對照表。
print(text, "→", result["label"], round(result["score"], 3))


**本節總結：**完整資料集 67,349 筆；預設抽樣 **10,000 筆** → Train **6,400**、Validation **1,600**、Test **2,000**。如果改成 `SAMPLE_SIZE = None`，會直接使用完整 CSV 的 67,349 筆，按相同影評分組切分，避免同一句跨組。以 Validation Loss 選最佳模型後，才評估 Test。

**資料限制：**影評原自英文 SST-2，再轉成繁體中文，部分句子可能有翻譯不自然或標註與語意不完全一致的情況；微調成果仍應檢查真實影評案例。
